# All Relevant Imports

In [19]:
import pandas as pd
import glob
import numpy as np


# Importing the Dataset

In [20]:


csv_files = sorted(glob.glob("./rawdata/*.csv"))
if len(csv_files) != 2:
	raise FileNotFoundError(f"Expected 2 CSV files in working dir, found {len(csv_files)}: {csv_files}")

fa24 = pd.read_csv(csv_files[0])
wi25 = pd.read_csv(csv_files[1])

print(f"Loaded: {csv_files[0]} -> fa24 (shape={fa24.shape}), {csv_files[1]} -> fa24 (shape={wi25.shape})\n")

print("fa24.head():")
print(fa24.head())
print("\nfa24.info():")
print(fa24.info())
print("\nfa24.describe():")
print(fa24.describe(include='all'))

print("\n---\n")
print("wi25.head():")
print(wi25.head())
print("\nwi25.info():")
print(wi25.info())
print("\nwi25.describe():")
print(wi25.describe(include='all'))

Loaded: ./rawdata/Master Sheet - All Events Fall_labeled_Final.csv -> fa24 (shape=(361, 14)), ./rawdata/Master Sheet - All Events Winter_Final.csv -> fa24 (shape=(245, 9))

fa24.head():
                              NAME_OF_EVENT       DATE  \
0                       Week 0 Meet & Greet  9/24/2024   
1                       Welcome Back Dinner  9/24/2024   
2  Chinese Union Fall 2024 Opening Ceremony  9/24/2024   
3                        Welcome Week Event  9/24/2024   
4                                 First GBM  9/25/2024   

                             VENUE     AWARDED A.S. Advertisement Pass  \
0         Student Center Courtyard    $587.24                       No   
1                Multipurpose Room  $1,752.15                       No   
2      Epstein Family Amphitheater  $7,931.74                       No   
3          Bear Room/Red Shoe Room    $600.00                       No   
4  Green Table Room (Price Center)    $296.31                       No   

   Estimated Attenda

### Helper Functions, taken from my prior project(Can be found at [Final Project – Group 049 WI25 (Jupyter Notebook)](https://github.com/COGS108/Group049_WI25/blob/master/FinalProject_Group049_WI25.ipynb))

In [21]:
def clean_awarded(funds):
    output = funds
    if funds == 'Price Center Red Shoe Room':
        output = np.nan
    elif isinstance(funds, str):
        output = output.strip()
        output = output.replace('$', '')
        output = output.replace(',', '')
        output = float(output)
    else: 
        output = str(funds)
        output = output.strip()
        output = output.replace('$', '')
        output = output.replace(',', '')
        output = float(output)
    return output

def clean_attendance(attendance):

    if attendance == '#VALUE!':
        output = np.nan
    elif attendance == '#DIV/0!':
        output = np.nan
    elif attendance == '#REF!':
        output = np.nan
    else:
        output = str(attendance)
        output = output.replace('%', '')
        output = float(output)

    return output



In [22]:
fa24

,NAME_OF_EVENT,DATE,VENUE,AWARDED,A.S. Advertisement Pass,Estimated Attendance,Form Attendance,PEEF Attendance,Attendance Miss,PEEF Miss,Status,Notes,Event_Type,Location_Type
0,Week 0 Meet & Greet,9/24/2024,Student Center Courtyard,$587.24,No,70,27.0,85,54.02%,-44.74%,Pending Warning,NaN,Social,Mandeville Auditorium
1,Welcome Back Dinner,9/24/2024,Multipurpose Room,"$1,752.15",No,152,95.0,183,45.78%,-4.44%,Pending Warning,NaN,Social,Mandeville Auditorium
2,Chinese Union Fall 2024 Opening Ceremony,9/24/2024,Epstein Family Amphitheater,"$7,931.74",No,800,NaN,800,100.00%,-0.86%,Pending Warning,NaN,Social,Pepper Canyon
3,Welcome Week Event,9/24/2024,Bear Room/Red Shoe Room,$600.00,No,80,NaN,0,100.00%,100.00%,Pending Warning,NaN,Social,Price Center
4,First GBM,9/25/2024,Green Table Room (Price Center),$296.31,No,30,NaN,70,100.00%,-136.24%,Pending Warning,NaN,GBM,Price Center
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
356,TWN Study Lodge: Winter Finals Prep Night,12/6/2024,Student Center - Dolores Huerta - Philip Vera ...,$200.00,NaN,0,8.0,0,60.00%,100.00%,NaN,NaN,Social,Mandeville Auditorium
357,End of Fall GBM,12/6/2024,Bear Room,$351.24,NaN,47,23.0,24,34.52%,31.67%,NaN,NaN,GBM,Price Center
358,IEEE Study Jam w/ Boba,12/6/2024,Warren College Room - Price,$246.75,NaN,40,24.0,0,2.74%,100.00%,NaN,NaN,Social,Price Center
359,Sun God Soccer Pickup Soccer,12/7/2024,Playing Fields(MUIR FIELD),$346.42,NaN,40,25.0,70,27.83%,-102.07%,NaN,NaN,Social,Muir


In [23]:
columns = ['VENUE', 'DATE', 'AWARDED', 'NAME_OF_EVENT', "A.S. Advertisement Pass", "Form Attendance", "Attendance Miss", "Event_Type",	"Location_Type", "Estimated Attendance"]
fa24 = fa24[columns]

# implement functions to clean columns
fa24['Attendance Miss'] = fa24['Attendance Miss'].apply(clean_attendance)
fa24['AWARDED'] = fa24['AWARDED'].apply(clean_awarded)
fa24 = fa24[fa24['Estimated Attendance'] != 0]

#add column called attendance_ratio for future EDA
fa24['Attendance_Ratio'] = 1 - fa24['Attendance Miss'].astype(float)/100

# Ensure your date column is named correctly (change 'Date' to your actual column name)
fa24['DATE'] = pd.to_datetime(fa24['DATE'])  # Convert to datetime


fa24events = fa24
fa24.head()

/tmp/ipykernel_7158/1291244194.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fa24['Attendance Miss'] = fa24['Attendance Miss'].apply(clean_attendance)
/tmp/ipykernel_7158/1291244194.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fa24['AWARDED'] = fa24['AWARDED'].apply(clean_awarded)


,VENUE,DATE,AWARDED,NAME_OF_EVENT,A.S. Advertisement Pass,Form Attendance,Attendance Miss,Event_Type,Location_Type,Estimated Attendance,Attendance_Ratio
0,Student Center Courtyard,2024-09-24,587.24,Week 0 Meet & Greet,No,27.0,54.02,Social,Mandeville Auditorium,70,0.4598
1,Multipurpose Room,2024-09-24,1752.15,Welcome Back Dinner,No,95.0,45.78,Social,Mandeville Auditorium,152,0.5422
2,Epstein Family Amphitheater,2024-09-24,7931.74,Chinese Union Fall 2024 Opening Ceremony,No,NaN,100.00,Social,Pepper Canyon,800,0.0000
3,Bear Room/Red Shoe Room,2024-09-24,600.00,Welcome Week Event,No,NaN,100.00,Social,Price Center,80,0.0000
4,Green Table Room (Price Center),2024-09-25,296.31,First GBM,No,NaN,100.00,GBM,Price Center,30,0.0000


In [24]:
# Get the columns we are interested in
columns = ['VENUE', 'DATE', 'AWARDED', 'NAME_OF_EVENT', "Form Attendance","A.S. Advertisement Pass", "Attendance Miss", "Event_Type",	"Location_Type"]

wi25 = wi25[columns]

# Implement functions to clean columns
wi25['Attendance Miss'] = wi25['Attendance Miss'].apply(clean_attendance)
wi25['AWARDED'] = wi25['AWARDED'].apply(clean_awarded)

# Add column called attendance_ratio for future EDA
wi25['Attendance_Ratio'] = 1 - wi25['Attendance Miss'].astype(float)/100

wi25['DATE'] = pd.to_datetime(wi25['DATE'])  # Convert to datetime
wi25events = wi25
wi25.head()

,VENUE,DATE,AWARDED,NAME_OF_EVENT,Form Attendance,A.S. Advertisement Pass,Attendance Miss,Event_Type,Location_Type,Attendance_Ratio
0,Red Shoe Room,2025-01-03,500.00,Pani Puri Event,NaN,Not Found,100.00,Religion,Price Center,0.0000
1,Price Center - Red Shoe Room,2025-01-06,350.00,Armenian Christmas,50.0,Not Found,-42.86,Religion,Price Center,1.4286
2,PC Ballroom East,2025-01-06,1483.03,GBM 1: Winter,NaN,Not Found,100.00,GBM,Price Center,0.0000
3,Price Center Ballroom B,2025-01-06,641.03,Board Game and Boba Winter Game Night # 1,131.0,No,-104.36,Social,Price Center,2.0436
4,Student Center - Dolores Huerta - Philip Vera ...,2025-01-07,400.00,MYSA GBM 2 Winter,28.0,Not Found,30.00,GBM,Mandeville Auditorium,0.7000


### Combining the 2 Datasets

In [25]:
# Find shared columns
shared_cols = list(set(fa24.columns).intersection(set(wi25.columns)))

# Drop unique columns
fa24_common = fa24[shared_cols].copy()
wi25_common = wi25[shared_cols].copy()

# Now you can safely concatenate
combined = pd.concat([fa24_common, wi25_common], ignore_index=True)
combined['Day_of_Week'] = combined['DATE'].dt.day_name()
combined = combined.drop(columns=["NAME_OF_EVENT","Attendance Miss", "Form Attendance", "VENUE"])

# Standardize A.S. Advertisement Pass column
combined['A.S. Advertisement Pass'] = (
    combined['A.S. Advertisement Pass']
    .astype(str)                                # convert to string to avoid NaN issues
    .str.strip()                                # remove spaces
    .str.lower()                                # normalize case
    .apply(lambda x: 'Yes' if 'yes' in x else 'No')
)

# Verify counts
print(combined['A.S. Advertisement Pass'].value_counts())

# Drop NaN and zero values in Attendance_Ratios
combined = combined.dropna(subset=["Attendance_Ratio"])
combined  = combined[combined ["Attendance_Ratio"] != 0]

print(f"\nCombined shape: {combined.shape}")

A.S. Advertisement Pass
No     469
Yes    126
Name: count, dtype: int64

Combined shape: (442, 7)


In [26]:
combined

,DATE,Location_Type,Event_Type,AWARDED,A.S. Advertisement Pass,Attendance_Ratio,Day_of_Week
0,2024-09-24,Mandeville Auditorium,Social,587.24,No,0.4598,Tuesday
1,2024-09-24,Mandeville Auditorium,Social,1752.15,No,0.5422,Tuesday
6,2024-09-25,Price Center,Social,590.47,No,0.7960,Wednesday
8,2024-09-26,Revelle,Social,407.30,No,1.5468,Thursday
10,2024-09-26,Price Center,Workshop,123.60,No,1.1327,Thursday
...,...,...,...,...,...,...,...
556,2025-02-17,Off Campus,Workshop,143.85,No,0.1390,Monday
557,2025-02-18,Price Center,GBM,200.00,No,0.8000,Tuesday
558,2025-02-18,Warren,Workshop,215.39,Yes,1.8107,Tuesday
561,2025-02-18,Price Center,Speaker,251.85,No,0.3574,Tuesday


In [27]:
combined.head()
combined.to_csv("./cleaneddata/combined_events.csv", index=False)